# Basic RAG Pipeline

This notebook builds a complete, end-to-end Retrieval-Augmented Generation pipeline using modern LangChain APIs.

**Pipeline**: Document Loading → Text Splitting → Embedding & Indexing → Retriever → Stuff Documents Chain → Retrieval Chain → invoke()

## Step 1: Document Loading

**Why TextLoader?** TextLoader reads plain text files into a single Document object with page_content and metadata (source path). It is the simplest loader for unstructured text.

**Common Mistake**: Not specifying encoding parameter for non-UTF8 files causes UnicodeDecodeError.

In [7]:
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import TextLoader

load_dotenv(find_dotenv())

# Load the speech document
loader = TextLoader("../documents/speech.txt", encoding="utf-8")
raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} document(s)")
print(f"Document length: {len(raw_documents[0].page_content)} characters")
print(f"Metadata: {raw_documents[0].metadata}")
print(f"Preview: {raw_documents[0].page_content[:200]}...")


Loaded 1 document(s)
Document length: 3624 characters
Metadata: {'source': '../documents/speech.txt'}
Preview: The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no ...


## Step 2: Text Splitting

**Why RecursiveCharacterTextSplitter?** It tries multiple separators in order (\n\n → \n → space → character) to preserve semantic boundaries like paragraphs and sentences.

**Why chunk_size=500, chunk_overlap=50?**
- 500 chars balances retrieval precision with context richness.
- 50 chars overlap prevents information loss at chunk boundaries.

**Common Mistake**: Using chunk_overlap=0 splits mid-sentence, losing cross-boundary context.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars | {chunk.page_content[:80]}...")


Split into 10 chunks
Chunk 1: 470 chars | The world must be made safe for democracy. Its peace must be planted upon the te...
Chunk 2: 350 chars | Just because we fight without rancor and without selfish object, seeking nothing...
Chunk 3: 498 chars | It will be all the easier for us to conduct ourselves as belligerents in a high ...
Chunk 4: 215 chars | and shall desire nothing so much as the early reestablishment of intimate relati...
Chunk 5: 496 chars | We have borne with their present government through all these bitter months beca...
Chunk 6: 499 chars | are in fact loyal to their neighbors and to the government in the hour of test. ...
Chunk 7: 79 chars | here and there and without countenance except from a lawless and malignant few....
Chunk 8: 498 chars | It is a distressing and oppressive duty, gentlemen of the Congress, which I have...
Chunk 9: 339 chars | always carried nearest our hearts—for democracy, for the right of those who subm...
Chunk 10: 355 chars | To such a task

## Step 3: Embedding & Vector Store Indexing

**Why GoogleGenerativeAIEmbeddings?** Uses Gemini embedding model (gemini-embedding-001, 3072 dimensions) for high-quality semantic representations. Works with your existing GOOGLE_API_KEY.

**Why Chroma (in-memory)?** Ephemeral in-memory mode avoids SQLite file lock issues during development. For production, use persist_directory or a dedicated vector database.

**Common Mistake**: Mixing different embedding models between indexing and querying causes dimension mismatch errors.

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# Initialize embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# Create in-memory Chroma vector store from document chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Indexed {len(chunks)} chunks into Chroma vector store")


Indexed 10 chunks into Chroma vector store


## Step 4: Create Retriever

**Why as_retriever()?** Converts the vector store into an LCEL Runnable component that accepts a string query and returns a list of Documents. This is required by create_retrieval_chain.

**Why k=3?** Returns the top 3 most relevant chunks. Start small and increase only if answer quality suffers from missing context.

In [10]:
# Create retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test retriever independently
test_docs = retriever.invoke("democracy and freedom")
print(f"Retriever returned {len(test_docs)} documents")
for i, doc in enumerate(test_docs, 1):
    print(f"--- Retrieved Chunk {i} ({len(doc.page_content)} chars) ---")
    print(f"{doc.page_content[:150]}...\n")


Retriever returned 3 documents
--- Retrieved Chunk 1 (339 chars) ---
always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the righ...

--- Retrieved Chunk 2 (339 chars) ---
always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the righ...

--- Retrieved Chunk 3 (470 chars) ---
The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serv...



## Step 5: Build RAG Chain (LCEL)

**Why create_stuff_documents_chain?** It takes the retrieved documents and "stuffs" them all into the prompt context. This is the simplest and most common strategy for RAG.

**Why create_retrieval_chain?** It composes the retriever and the stuff chain into a single end-to-end pipeline: query → retrieve → stuff → generate.

**Why invoke() not predict()?** predict() is deprecated. invoke() is the modern LCEL standard that supports streaming, batching, and async.

**Common Mistake**: Using the wrong prompt variable names. create_stuff_documents_chain expects {context} for documents and {input} for the user query.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# Define prompt template
# IMPORTANT: Variables must be named 'context' and 'input' for create_stuff_documents_chain
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based on the provided context.
Use ONLY the following context to answer the question.
If the context does not contain enough information to answer, say: "I don't have enough information to answer this question."

Context:
{context}"""),
    ("human", "{input}")
])

# Step 1: Create stuff documents chain (combines retrieved docs into prompt)
combine_docs_chain = create_stuff_documents_chain(llm, prompt)

# Step 2: Create retrieval chain (retriever + stuff chain = full RAG pipeline)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

print("RAG chain built successfully!")
print(f"Chain type: {type(rag_chain).__name__}")


RAG chain built successfully!
Chain type: RunnableBinding


## Step 6: Query the RAG Chain

The retrieval chain returns a dictionary with:
- result["input"]: The original user query.
- result["context"]: The list of retrieved Document objects.
- result["answer"]: The LLM-generated answer grounded in retrieved context.

In [12]:
# Query 1: Direct factual question
result = rag_chain.invoke({"input": "What is the main message about democracy in this speech?"})

print("=== Query 1 ===")
print(f"Question: {result['input']}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources Retrieved: {len(result['context'])} chunks")
for i, doc in enumerate(result['context'], 1):
    print(f"  Source {i}: {doc.metadata.get('source', 'unknown')} ({len(doc.page_content)} chars)")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Query 1 ===
Question: What is the main message about democracy in this speech?

Answer: Based on the provided text, the main message about democracy includes the following key points:

* **The world must be made safe for democracy**, and its peace must be built on the "tested foundations of political liberty."
* Democracy represents **"the right of those who submit to authority to have a voice in their own governments."**
* It is one of the principles described as being **"always carried nearest our hearts."**

Sources Retrieved: 3 chunks
  Source 1: ../documents/speech.txt (339 chars)
  Source 2: ../documents/speech.txt (339 chars)
  Source 3: ../documents/speech.txt (470 chars)


In [13]:
# Query 2: Question about a specific detail
result2 = rag_chain.invoke({"input": "What attitude does the speaker express toward the German people?"})

print("=== Query 2 ===")
print(f"Question: {result2['input']}")
print(f"\nAnswer: {result2['answer']}")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Query 2 ===
Question: What attitude does the speaker express toward the German people?

Answer: Based on the provided context, the speaker expresses an attitude of friendship toward the German people. Specifically, the speaker states that they:

* Are "sincere friends of the German people."
* Act "without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them."
* Wish to prove that friendship in their "daily attitude and actions" toward people of German birth and sympathy who live among them and are loyal.


In [14]:
# Query 3: Out-of-context question (should trigger fallback)
result3 = rag_chain.invoke({"input": "What is the capital of France?"})

print("=== Query 3 (Out-of-Context Test) ===")
print(f"Question: {result3['input']}")
print(f"\nAnswer: {result3['answer']}")


/Users/kapilyadav/Coding_Space/Python_workspace/LangChainWorkspace/generativeai/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Query 3 (Out-of-Context Test) ===
Question: What is the capital of France?

Answer: I don't have enough information to answer this question.
